# GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes

Fully self-contained GNN+MNN notebook. Runs end-to-end:

1. Load CAMELS-LH halos (one graph per halo) for **Astrid, IllustrisTNG, SIMBA, SWIFT-EAGLE**.
2. Attach photometry (ugriz + UKIDSS Y).
3. Build PyG `Data` objects with positions, stellar mass, |vel|, magnitudes.
4. Train an EdgeNet GNN with a 2-moment MNN loss.
5. Save the model, predict (μ, σ) per test halo, save predictions.
6. Report metrics, residual plots, mass-binned diagnostics, TARP coverage.

 Stellar mass, |vel|, and five ugriz+Y magnitudes (plus g-r, r-i colors) are used as per-galaxy node features.


In [ ]:
import os
import sys
import glob
import time
import json

import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# HaloGraphNet's ModelGNN lives one directory up in the original checkout.
REPO_ROOT = os.path.abspath("../..")
HGN_ROOT  = os.path.join(REPO_ROOT, "HaloGraphNet")
if HGN_ROOT not in sys.path:
    sys.path.insert(0, HGN_ROOT)

# with the installed torch (common on macOS with very new torch + cp313).

# ---- torch_scatter / torch_cluster ABI workaround ----
# Some torch_scatter / torch_cluster wheels are ABI-incompatible with
# the installed PyTorch (notably on macOS + Python 3.13). HaloGraphNet
# only uses a handful of symbols, which we polyfill onto pure torch +
# torch_geometric so this notebook runs unchanged either way.
import sys as _sys, types as _types

def _torch_scatter_works():
    try:
        import torch_scatter as _ts
        _ts.scatter_mean(torch.zeros(2, 1), torch.tensor([0, 1]), dim=0)
        return True
    except Exception:
        return False

if not _torch_scatter_works():
    from torch_geometric.utils import scatter as _pyg_scatter
    _ts_mod = _types.ModuleType("torch_scatter")
    def _make(reduce, return_tuple):
        def fn(src, index, dim=0, out=None, dim_size=None):
            res = _pyg_scatter(src, index, dim=dim, dim_size=dim_size, reduce=reduce)
            return (res, None) if return_tuple else res
        return fn
    _ts_mod.scatter_mean = _make("mean", False)
    _ts_mod.scatter_sum  = _make("sum",  False)
    _ts_mod.scatter_max  = _make("max",  True)
    _ts_mod.scatter_min  = _make("min",  True)
    _sys.modules["torch_scatter"] = _ts_mod
    print("(installed torch_scatter polyfill)")

# torch_cluster: native ABI mismatches surface as SIGBUS, which kills
# the kernel. Always replace with a pure-torch fallback (slower but
# correct; halos rarely have >200 galaxies so this is fine).
_tc_mod = _types.ModuleType("torch_cluster")

def _radius_graph(pos, r, batch=None, loop=False, max_num_neighbors=32,
                  flow="source_to_target", num_workers=1):
    N = pos.size(0); device = pos.device
    if batch is None:
        batch = torch.zeros(N, dtype=torch.long, device=device)
    d2 = torch.cdist(pos, pos, p=2)
    same = batch.unsqueeze(0) == batch.unsqueeze(1)
    within = (d2 <= r) & same
    if not loop:
        within = within & ~torch.eye(N, dtype=torch.bool, device=device)
    if max_num_neighbors is not None and max_num_neighbors < N:
        d2_m = torch.where(within, d2, torch.full_like(d2, float("inf")))
        kk = min(max_num_neighbors, N)
        kth = torch.topk(d2_m, k=kk, dim=1, largest=False).values
        thresh = kth[:, -1:].clamp(max=r)
        within = within & (d2 <= thresh)
    src, dst = within.nonzero(as_tuple=True)
    if flow == "target_to_source":
        return torch.stack([dst, src], dim=0)
    return torch.stack([src, dst], dim=0)

def _knn_graph(x, k, batch=None, loop=False, flow="source_to_target",
               cosine=False, num_workers=1):
    N = x.size(0); device = x.device
    if batch is None:
        batch = torch.zeros(N, dtype=torch.long, device=device)
    if cosine:
        xn = torch.nn.functional.normalize(x, dim=-1)
        d2 = 1.0 - xn @ xn.T
    else:
        d2 = torch.cdist(x, x, p=2)
    same = batch.unsqueeze(0) == batch.unsqueeze(1)
    d2 = torch.where(same, d2, torch.full_like(d2, float("inf")))
    if not loop:
        d2.fill_diagonal_(float("inf"))
    kk = min(k, N)
    nn_idx = torch.topk(d2, k=kk, dim=1, largest=False).indices
    src = torch.arange(N, device=device).unsqueeze(1).expand(-1, kk).reshape(-1)
    dst = nn_idx.reshape(-1)
    if flow == "target_to_source":
        return torch.stack([dst, src], dim=0)
    return torch.stack([src, dst], dim=0)

_tc_mod.radius_graph = _radius_graph
_tc_mod.knn_graph    = _knn_graph
_sys.modules["torch_cluster"] = _tc_mod

from Source.networks import ModelGNN
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader

plt.rcParams.update({"font.size": 11, "figure.figsize": (9, 5.5)})
np.set_printoptions(precision=4, suppress=True)

SPLIT_SEED = 42
np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



In [ ]:
# ---- Paths + constants ----
NB_DIR  = os.getcwd()
OUT_DIR = NB_DIR
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)

SIM_DIRS = {
    "Astrid": "data",
    "IllustrisTNG": "IllustrisTNG",
    "SIMBA": "SIMBA",
    "SWIFT-EAGLE": "Swift"
}
PHOTO_PREFIX = {
    "Astrid": "Astrid",
    "IllustrisTNG": "IllustrisTNG",
    "SIMBA": "Simba",
    "SWIFT-EAGLE": "Swift-EAGLE"
}

PHOTO_BANDS = [("SLOAN", "SDSS.g"), ("SLOAN", "SDSS.r"),
               ("SLOAN", "SDSS.i"), ("SLOAN", "SDSS.z"),
               ("UKIRT", "UKIDSS.Y")]
PHOTO_DEFAULTS = dict(snap="090", stellar_pop="BC03",
                      kind="luminosity", variant="attenuated")

MASS_CUT = 1.3e8
BOXSIZE  = 25_000.0    # kpc/h (CAMELS LH box)
VELNORM  = 100.0
RADNORM  = 8.0
MAG_REF  = 30.0

USE_PHOT = True
USE_VEL  = True
USE_HMR  = False

print(f"simulations: {list(SIM_DIRS)}  (with_photometry={USE_PHOT})")


In [ ]:
# ============================================================
# LOAD HALOS (one graph per halo, with all member galaxies)
# ============================================================
# For every halo that has at least one galaxy above the stellar-mass
# cut, we record all passing member galaxies' positions, velocities,
# stellar masses, and photometric luminosities (if requested).

def _lh_index(path: str) -> int:
    base = os.path.basename(path)
    if base == "groups_090.hdf5":
        return 0
    return int(base.split("(")[1].split(")")[0])


def _photo_path(sim: str, lh: int) -> str:
    sim_dir = SIM_DIRS[sim]
    prefix  = PHOTO_PREFIX[sim]
    return os.path.join(REPO_ROOT, "photometry", sim_dir, f"{prefix}_LH_{lh}_photometry.hdf5")


# Each halo entry: dict with pos/vel/mass/rad/phot, plus halo-level fields
halos = []
print("Loading halos ...")
for sim, sim_dir in SIM_DIRS.items():
    pattern = os.path.join(REPO_ROOT, sim_dir, "groups_090*.hdf5")
    files = sorted(glob.glob(pattern))
    print(f"  [{sim:12s}]  {len(files):2d} realisations")
    for path in files:
        lh = _lh_index(path)
        # Try to match a photometry file (optional).
        phot_L_by_sub = None
        if USE_PHOT:
            pp = _photo_path(sim, lh)
            if os.path.exists(pp):
                phot_L_by_sub = {}
                with h5py.File(pp, "r") as pf:
                    sub_idx = pf["subhalo_index"][:] if "subhalo_index" in pf else None
                    n_bands = len(PHOTO_BANDS)
                    for b, (family, band) in enumerate(PHOTO_BANDS):
                        grp = f"snap_{PHOTO_DEFAULTS['snap']}/{PHOTO_DEFAULTS['stellar_pop']}" \
                              f"/photometry/{PHOTO_DEFAULTS['kind']}/{PHOTO_DEFAULTS['variant']}" \
                              f"/{family}/{band}"
                        if grp in pf:
                            L = pf[grp][:]
                            if sub_idx is None:
                                for si, lum in enumerate(L):
                                    phot_L_by_sub.setdefault(int(si), np.full(n_bands, np.nan))
                                    phot_L_by_sub[int(si)][b] = lum
                            else:
                                for r, si in enumerate(sub_idx):
                                    phot_L_by_sub.setdefault(int(si), np.full(n_bands, np.nan))
                                    if r < len(L):
                                        phot_L_by_sub[int(si)][b] = L[r]

        with h5py.File(path, "r") as f:
            group_first_sub = f["Group/GroupFirstSub"][:]
            group_n_subs    = f["Group/GroupNsubs"][:]
            halo_mass       = f["Group/Group_M_Crit200"][:]    # 1e10 Msun/h
            subhalo_mass_t  = f["Subhalo/SubhaloMassType"][:]
            stellar_mass    = subhalo_mass_t[:, 4] * 1e10
            galaxy_radius_t = f["Subhalo/SubhaloHalfmassRadType"][:, 4]
            galaxy_pos      = f["Subhalo/SubhaloPos"][:]
            galaxy_vel      = f["Subhalo/SubhaloVel"][:]

            n_halos = len(group_first_sub)
            for h_idx in range(n_halos):
                if group_first_sub[h_idx] == -1:
                    continue
                first_sub = group_first_sub[h_idx]
                n_subs    = group_n_subs[h_idx]
                sub_indices = np.arange(first_sub, first_sub + n_subs)
                mass_mask = stellar_mass[sub_indices] > MASS_CUT
                valid = sub_indices[mass_mask]
                if len(valid) == 0:
                    continue

                entry = dict(
                    simulation=sim, lh_index=lh, halo_id=int(h_idx),
                    galaxy_ids=np.array(valid, dtype=np.int32),
                    halo_pos=np.array(galaxy_pos[first_sub], dtype=np.float32),
                    halo_vel=np.array(galaxy_vel[first_sub], dtype=np.float32),
                    galaxy_pos=np.array(galaxy_pos[valid], dtype=np.float32),
                    galaxy_vel=np.array(galaxy_vel[valid], dtype=np.float32),
                    log_stellar_mass=np.log10(stellar_mass[valid]).astype(np.float32),
                    log_galaxy_radius=np.log10(galaxy_radius_t[valid] + 1e-12).astype(np.float32),
                    log_halo_mass=float(np.log10(halo_mass[h_idx] * 1e10)),
                    log_luminosity=None,
                )
                if USE_PHOT and phot_L_by_sub is not None:
                    L = np.full((len(valid), len(PHOTO_BANDS)), np.nan, dtype=np.float32)
                    for j, gid in enumerate(valid):
                        row = phot_L_by_sub.get(int(gid))
                        if row is not None:
                            L[j] = row
                    # log10(L) where L>0; NaN otherwise
                    with np.errstate(invalid="ignore", divide="ignore"):
                        entry["log_luminosity"] = np.log10(L)
                halos.append(entry)

print(f"\nTotal halos loaded: {len(halos):,}")
print(f"Node-count stats: median={int(np.median([len(h['galaxy_ids']) for h in halos]))}"
      f"  max={int(np.max([len(h['galaxy_ids']) for h in halos]))}")


In [ ]:
# ============================================================
# CONVERT EACH HALO TO A PyG Data OBJECT
# ============================================================
# Node features: (pos_x, pos_y, pos_z, log M*, log |v|[, magnitudes, g-r, r-i])
# Edges are rebuilt by the GNN's radius_graph on ``pos`` at each forward
# pass — we just store positions and features per node.

def _correct_boundary(pos: np.ndarray, box: float = 1.0) -> np.ndarray:
    pos = np.asarray(pos, dtype=np.float32).copy()
    pos[pos >  box / 2] -= box
    pos[pos < -box / 2] += box
    return pos


def halo_to_data(halo: dict) -> Data:
    sm  = halo["log_stellar_mass"]
    pos = halo["galaxy_pos"] / BOXSIZE
    vel = halo["galaxy_vel"] / VELNORM
    ref_p = halo["halo_pos"] / BOXSIZE
    ref_v = halo["halo_vel"] / VELNORM
    pos = _correct_boundary(pos - ref_p)
    vel = vel - ref_v

    cols = [pos.astype(np.float32), sm.astype(np.float32)[:, None]]
    if USE_VEL:
        vmag = np.log10(np.sqrt(np.sum(vel ** 2, axis=1)) + 1e-12)
        cols.append(vmag.astype(np.float32)[:, None])

    if USE_PHOT and halo["log_luminosity"] is not None:
        L = 10 ** halo["log_luminosity"]
        with np.errstate(invalid="ignore", divide="ignore"):
            m = (-2.5 * np.log10(L) - MAG_REF).astype(np.float32)
        for j in range(m.shape[1]):
            col = m[:, j]
            bad = ~np.isfinite(col)
            if bad.any():
                good = col[~bad]
                fill = float(np.median(good)) if good.size else 0.0
                col[bad] = fill
                m[:, j] = col
        cols.append(m)
        if m.shape[1] >= 3:
            gr = (m[:, 0] - m[:, 1])[:, None]
            ri = (m[:, 1] - m[:, 2])[:, None]
            cols.append(gr); cols.append(ri)

    x = np.concatenate(cols, axis=1).astype(np.float32)
    u = np.zeros((1, 2), dtype=np.float32)
    u[0, 0] = x.shape[0]
    u[0, 1] = float(np.log10(np.sum(10 ** sm)))

    return Data(
        x=torch.tensor(x, dtype=torch.float32),
        pos=torch.tensor(pos, dtype=torch.float32),
        y=torch.tensor(halo["log_halo_mass"], dtype=torch.float32),
        u=torch.tensor(u, dtype=torch.float32),
    )


pyg_list = [halo_to_data(h) for h in halos]
node_features = int(pyg_list[0].x.shape[1])
print(f"node_features per graph: {node_features}")


In [ ]:
# ============================================================
# TRAIN / VAL / TEST SPLIT  (80/20, with val = 10% of train)
# ============================================================
n = len(pyg_list)
rng = np.random.default_rng(SPLIT_SEED)
perm = rng.permutation(n)
n_train = int(0.8 * n)
train_idx = perm[:n_train]
test_idx  = perm[n_train:]
val_size  = max(1, int(0.1 * len(train_idx)))
val_idx   = train_idx[:val_size]
train_idx = train_idx[val_size:]

train_data = [pyg_list[i] for i in train_idx]
val_data   = [pyg_list[i] for i in val_idx]
test_data  = [pyg_list[i] for i in test_idx]
print(f"n={n:,}  train={len(train_data):,}  val={len(val_data):,}  test={len(test_data):,}")


In [ ]:
# ============================================================
# MODEL:  HaloGraphNet EdgeNet + Moment Neural Network head
# ============================================================
arch = dict(
    use_model="EdgeNet",
    node_features=node_features,
    n_layers=2,
    k_nn=5.0,
    hidden_channels=300,
    latent_channels=100,
)
model = ModelGNN(**arch).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"arch: {arch}")
print(f"params: {n_params:,}")


In [ ]:
# ============================================================
# TRAINING  (2-moment MNN loss)
# ============================================================
from scipy.spatial.transform import Rotation as Rot

batch_size    = 128
n_epochs      = 100
learning_rate = 0.001
weight_decay  = 1e-05
AUGMENT       = True

opt = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
train_loader = PyGDataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = PyGDataLoader(val_data,   batch_size=batch_size, shuffle=False)


def moment_loss(out: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    y_out, err_out = out[:, 0], out[:, 1]
    loss_mse = torch.mean((y_out - y) ** 2)
    loss_lfi = torch.mean(((y_out - y) ** 2 - err_out ** 2) ** 2)
    return torch.log(loss_mse + 1e-12) + torch.log(loss_lfi + 1e-12)


def augment_rotation(data):
    R = torch.tensor(Rot.random().as_matrix(), dtype=torch.float32)
    data.pos = data.pos @ R.T
    data.x[:, :3] = data.x[:, :3] @ R.T
    return data


best = float("inf"); best_epoch = -1; best_state = None
train_losses, val_losses = [], []
t0 = time.time()
for epoch in range(1, n_epochs + 1):
    # train
    model.train()
    total, nb = 0.0, 0
    for data in train_loader:
        if AUGMENT:
            data = augment_rotation(data)
        data = data.to(device)
        opt.zero_grad()
        loss = moment_loss(model(data), data.y.view(-1))
        loss.backward(); opt.step()
        total += loss.item(); nb += 1
    train_losses.append(total / max(1, nb))

    # val
    model.eval()
    total_v, nb_v = 0.0, 0
    with torch.no_grad():
        for data in val_loader:
            data = data.to(device)
            total_v += moment_loss(model(data), data.y.view(-1)).item(); nb_v += 1
    val_losses.append(total_v / max(1, nb_v))

    if val_losses[-1] < best:
        best, best_epoch = val_losses[-1], epoch
        best_state = {k: v.clone().detach().cpu() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % max(1, n_epochs // 10) == 0:
        print(f"epoch {epoch:3d}/{n_epochs} | train {train_losses[-1]:+.3f}"
              f" | val {val_losses[-1]:+.3f} | best {best:+.3f} @ {best_epoch}")

# Restore best
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nTraining done in {time.time() - t0:.1f}s  best val {best:+.3f} @ epoch {best_epoch}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(train_losses, label="train")
ax.plot(val_losses, label="val")
ax.axvline(best_epoch, ls="--", color="k", lw=1, label=f"best @ {best_epoch}")
ax.set_xlabel("epoch"); ax.set_ylabel("MNN loss"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes — training curves")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "training_curves.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# SAVE CHECKPOINT
# ============================================================
model_path = os.path.join(OUT_DIR, "best_model.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "arch":     arch,
    "features": dict(use_hmR=0, use_vel=int(USE_VEL), use_phot=int(USE_PHOT),
                     galcen_frame=0, only_positions=0),
    "norms":    dict(boxsize=BOXSIZE, velnorm=VELNORM, radnorm=RADNORM, mag_ref=MAG_REF),
    "best_val_loss": float(best),
    "best_epoch":    int(best_epoch),
    "model_name":    "GNN+MNN (MEGA) \u2013 stellar mass + |vel| + magnitudes",
    "y_offset":      0.0,
}, model_path)
print(f"saved -> {model_path}")


In [ ]:
# ============================================================
# INFERENCE  (moment predict on test set)
# ============================================================
n_posterior = 1000
test_loader = PyGDataLoader(test_data, batch_size=64, shuffle=False)

mu_all, sd_all, y_all = [], [], []
model.eval()
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out  = model(data)
        mu_all.append(out[:, 0].cpu().numpy())
        sd_all.append(out[:, 1].abs().cpu().numpy())
        y_all.append(data.y.view(-1).cpu().numpy())
mu = np.concatenate(mu_all)
sd = np.concatenate(sd_all)
y_true = np.concatenate(y_all)

rng = np.random.default_rng(0)
samples = rng.normal(loc=mu[:, None], scale=np.maximum(sd[:, None], 1e-6),
                     size=(len(mu), n_posterior)).astype(np.float32)

print(f"test predictions: μ range [{mu.min():.2f}, {mu.max():.2f}]"
      f"  truth range [{y_true.min():.2f}, {y_true.max():.2f}]")


In [ ]:
# ============================================================
# SAVE POSTERIOR PREDICTIONS
# ============================================================
sim_arr     = np.array([halos[i]["simulation"] for i in test_idx])
halo_id_arr = np.array([halos[i]["halo_id"]    for i in test_idx], dtype=np.int32)
lh_arr      = np.array([halos[i]["lh_index"]   for i in test_idx], dtype=np.int32)
gal_ids_arr = np.array([halos[i]["galaxy_ids"] for i in test_idx], dtype=object)

pred_path = os.path.join(OUT_DIR, "test_predictions.npz")
np.savez_compressed(
    pred_path,
    simulation=sim_arr,
    halo_id=halo_id_arr,
    galaxy_ids=gal_ids_arr,
    lh_index=lh_arr,
    true_log_halo_mass=y_true.astype(np.float32),
    posterior_mean=mu.astype(np.float32),
    posterior_std=sd.astype(np.float32),
    posterior_log_halo_mass=samples,
    n_posterior=int(n_posterior),
    model_name="GNN+MNN (MEGA) \u2013 stellar mass + |vel| + magnitudes",
)
print(f"saved -> {pred_path}")


In [ ]:
# ============================================================
# METRICS (RMSE, chi2, PCC, eps, R^2)
# ============================================================
def compute_metrics(y, mu, sigma):
    y, mu, sigma = map(np.asarray, (y, mu, sigma))
    residual = mu - y
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    chi2 = float(np.mean((residual / np.maximum(sigma, 1e-6)) ** 2))
    pcc  = float(np.corrcoef(y, mu)[0, 1]) if y.std() > 0 and mu.std() > 0 else float("nan")
    eps  = float(np.mean(np.abs(10 ** mu - 10 ** y) / (10 ** y)) * 100.0)
    ss_res = float(np.sum(residual ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    return dict(rmse=rmse, chi2=chi2, pcc=pcc, eps=eps, r2=r2)

overall = compute_metrics(y_true, mu, sd)
print(f"Overall (N={len(y_true)}):")
for k, v in overall.items():
    print(f"  {k:4s} = {v:.4f}")

print("\n--- Per simulation ---")
for s in sorted(set(sim_arr.tolist())):
    sel = sim_arr == s
    if sel.sum() == 0:
        continue
    m = compute_metrics(y_true[sel], mu[sel], sd[sel])
    print(f"  {s:14s}  N={sel.sum():5d}  RMSE={m['rmse']:.4f}  chi2={m['chi2']:.2f}"
          f"  PCC={m['pcc']:.3f}  eps={m['eps']:.1f}%  R2={m['r2']:.3f}")



In [ ]:
# ============================================================
# SAVE metrics.json + hyperparameters.txt
# ============================================================
metrics_payload = {"model": "GNN+MNN (MEGA) \u2013 stellar mass + |vel| + magnitudes", "overall": overall, "per_simulation": {}}
for s in sorted(set(sim_arr.tolist())):
    sel = sim_arr == s
    if sel.sum() == 0:
        continue
    metrics_payload["per_simulation"][s] = compute_metrics(
        y_true[sel], mu[sel], sd[sel]
    )
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as fh:
    json.dump(metrics_payload, fh, indent=2)
print(f"saved -> {os.path.join(OUT_DIR, 'metrics.json')}")

hparams = {
    "model_name":     "GNN+MNN (MEGA) \u2013 stellar mass + |vel| + magnitudes",
    "use_model":      arch["use_model"],
    "node_features":  int(node_features),
    "n_layers":       arch["n_layers"],
    "k_nn":           arch["k_nn"],
    "hidden_channels": arch["hidden_channels"],
    "latent_channels": arch["latent_channels"],
    "use_vel":        bool(USE_VEL),
    "use_phot":       bool(USE_PHOT),
    "boxsize":        BOXSIZE,
    "velnorm":        VELNORM,
    "mag_ref":        MAG_REF,
    "learning_rate":  float(learning_rate),
    "weight_decay":   float(weight_decay),
    "batch_size":     int(batch_size),
    "n_epochs":       int(n_epochs),
    "augment_rotations": bool(AUGMENT),
    "split_seed":     int(SPLIT_SEED),
    "n_train":        int(len(train_data)),
    "n_val":          int(len(val_data)),
    "n_test":         int(len(test_data)),
    "best_val_loss":  float(best),
    "best_epoch":     int(best_epoch),
}
with open(os.path.join(OUT_DIR, "hyperparameters.txt"), "w") as fh:
    fh.write(f"# Hyperparameters for {hparams['model_name']}\n")
    for k, v in hparams.items():
        fh.write(f"{k}: {v}\n")
print(f"saved -> {os.path.join(OUT_DIR, 'hyperparameters.txt')}")


In [ ]:
# ============================================================
# RESIDUAL PANEL (combined)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_true, mu - y_true, s=6, alpha=0.3)
axes[0].axhline(0, color="k", lw=1)
axes[0].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[0].set_ylabel("Predicted − True")
axes[0].set_title(f"GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes — residuals (N={len(y_true)})")
axes[0].grid(alpha=0.3)

axes[1].hist(mu - y_true, bins=60, color="steelblue", alpha=0.8, edgecolor="k")
axes[1].axvline(0, color="k", lw=1)
mu_r = float(np.mean(mu - y_true)); sd_r = float(np.std(mu - y_true))
axes[1].set_xlabel("Predicted − True")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Residual dist  (μ={mu_r:+.3f}, σ={sd_r:.3f})")
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "residuals_combined.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# PER-SIMULATION RESIDUAL GRID
# ============================================================
sims = sorted(set(sim_arr.tolist()))
n = len(sims)
cols = min(n, 2); rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.2 * rows),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes).flatten()
for ax, s in zip(axes, sims):
    sel = sim_arr == s
    if sel.sum() == 0:
        ax.set_visible(False); continue
    ax.scatter(y_true[sel], mu[sel] - y_true[sel], s=6, alpha=0.35)
    ax.axhline(0, color="k", lw=1)
    m = compute_metrics(y_true[sel], mu[sel], sd[sel])
    ax.set_title(f"{s}  N={sel.sum():,}  RMSE={m['rmse']:.3f}  chi²={m['chi2']:.2f}")
    ax.set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
    ax.set_ylabel("Predicted − True")
    ax.grid(alpha=0.3)
for ax in axes[len(sims):]:
    ax.set_visible(False)
fig.suptitle(f"GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes — per simulation", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "residuals_per_simulation.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# METRICS IN MASS BINS
# ============================================================
def metrics_in_bins(y, mu, sigma, n_bins=10):
    edges   = np.linspace(y.min(), y.max(), n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    rmse_b, chi_b = [], []
    for i in range(n_bins):
        sel = (y >= edges[i]) & (y < edges[i + 1])
        if sel.sum() < 2:
            rmse_b.append(np.nan); chi_b.append(np.nan); continue
        rmse_b.append(np.sqrt(np.mean((mu[sel] - y[sel]) ** 2)))
        chi_b.append(np.mean(((mu[sel] - y[sel]) / np.maximum(sigma[sel], 1e-6)) ** 2))
    return centers, np.array(rmse_b), np.array(chi_b)

centers, rmse_b, chi_b = metrics_in_bins(y_true, mu, sd)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(centers, rmse_b, "o-", color="steelblue")
axes[0].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[0].set_ylabel("RMSE"); axes[0].grid(alpha=0.3)
axes[0].set_title("RMSE in halo-mass bins")
axes[1].plot(centers, chi_b, "o-", color="darkorange")
axes[1].axhline(1.0, color="k", ls="--", lw=1)
axes[1].set_xlabel(r"True $\log_{10}\,M_{\rm halo}$")
axes[1].set_ylabel(r"$\chi^2$"); axes[1].grid(alpha=0.3)
axes[1].set_title(r"$\chi^2$ in halo-mass bins")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "plots", "metrics_vs_mass.png"), dpi=150)
plt.show()


In [ ]:
# ============================================================
# TARP COVERAGE TEST
# ============================================================
try:
    import tarp
    tarp_samples = samples.T[:, :, np.newaxis]           # (n_post, N_test, 1)
    theta        = y_true[:, np.newaxis]                 # (N_test, 1)
    ecp, alpha = tarp.get_tarp_coverage(
        tarp_samples, theta,
        references="random", metric="euclidean",
        num_bootstrap=100, bootstrap=True, seed=0,
    )
    if ecp.ndim == 2:
        ecp_mean = ecp.mean(axis=0)
    else:
        ecp_mean = ecp
    d = np.max(np.abs(ecp_mean - alpha))
    ks_p = float(np.exp(-2 * len(alpha) * d ** 2))

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot(alpha, alpha, "k--", lw=1, label="perfect")
    if ecp.ndim == 2:
        lo, hi = np.quantile(ecp, [0.16, 0.84], axis=0)
        ax.fill_between(alpha, lo, hi, color="C0", alpha=0.25, label="68%")
        ax.plot(alpha, ecp_mean, color="C0", lw=2)
    else:
        ax.plot(alpha, ecp, color="C0", lw=2)
    ax.set_xlabel("Credibility α")
    ax.set_ylabel("Expected coverage prob.")
    ax.text(0.04, 0.96, f"KS p = {ks_p:.3g}", transform=ax.transAxes,
            va="top", bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    ax.set_title(f"GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes — TARP coverage")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "plots", "tarp_coverage.png"), dpi=150)
    plt.show()
except Exception as e:
    print(f"TARP failed: {e}")


## Summary — GNN+MNN (MEGA) – stellar mass + |vel| + magnitudes

Artifacts written alongside this notebook:

- `best_model.pt` — `ModelGNN` weights, arch, feature flags, normalisers
- `test_predictions.npz` — (μ, σ) per test halo, 1000 Gaussian posterior draws, `simulation`, `halo_id`, `galaxy_ids` (per-halo, object array), `lh_index`, `true_log_halo_mass`
- `plots/training_curves.png`, `plots/residuals_combined.png`, `plots/residuals_per_simulation.png`, `plots/metrics_vs_mass.png`, `plots/tarp_coverage.png`
